In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import matplotlib.pyplot as plt

from GP import *

In [ ]:
# ===== generate training data =====
torch.manual_seed(0)

n = 20
X = torch.linspace(-5, 5, n).unsqueeze(1)   # (n, 1)

true_function = torch.sin(X)
noise = 0.2 * torch.randn_like(X)

y = true_function + noise   # (n, 1)

# GP

In [ ]:
gp = GP(kernel="rbf")

optimizer = torch.optim.Adam(gp.parameters(), lr=0.05)

for epoch in range(200):
    optimizer.zero_grad()
    
    loss = -gp.log_marginal_likelihood(X, y)  # maximize → minimize negative
    
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [ ]:
# ===== test points =====
X_star = torch.linspace(-6, 6, 200).unsqueeze(1)

with torch.no_grad():
    mu_star, sigma_star = gp.predict(X, y, X_star)

# 转 numpy 用于画图
X_np = X.numpy()
y_np = y.numpy()

X_star_np = X_star.numpy()
mu_np = mu_star.numpy()
std_np = torch.sqrt(torch.diag(sigma_star)).numpy()

# ===== plot =====
plt.figure(figsize=(8, 5))

# training points
plt.scatter(X_np, y_np, color='black', label='Data')

# mean
plt.plot(X_star_np, mu_np, label='Mean')

# confidence interval
plt.fill_between(
    X_star_np.flatten(),
    mu_np.flatten() - 2 * std_np,
    mu_np.flatten() + 2 * std_np,
    alpha=0.3,
    label='Confidence (2σ)'
)

plt.legend()
plt.title("Gaussian Process Regression (PyTorch)")
plt.show()